In [ ]:
import os
import numpy as np
import pandas as pd

# Paths
save_path = "data/processed"
os.makedirs(save_path, exist_ok=True)
factor_path = os.path.join(save_path, "factor_library.parquet")
adj_path = os.path.join(save_path, "graph_adj.npy")

# Load factor data
factors = pd.read_parquet(factor_path)
print(f"Loaded factor data with {factors['ticker'].nunique()} tickers and {len(factors)} rows.")

# Create ticker to sector mapping inline
ticker_sector = {
    "AAPL": "Tech",
    "AMZN": "Consumer",
    "GOOG": "Tech",
    "MSFT": "Tech"
}
print(f"Ticker to sector mapping: {ticker_sector}")

# Pivot factor returns to wide format: rows=date, columns=ticker, values=ret
ret_wide = factors.pivot_table(index="date", columns="ticker", values="ret", aggfunc='mean')

print(f"Return data shape (dates x tickers): {ret_wide.shape}")

# Compute correlation matrix between tickers based on returns
corr = ret_wide.corr()
print("Correlation matrix stats:")
print(f"  Mean corr: {corr.values[np.triu_indices_from(corr, k=1)].mean():.4f}")
print(f"  Min corr: {corr.values[np.triu_indices_from(corr, k=1)].min():.4f}")
print(f"  Max corr: {corr.values[np.triu_indices_from(corr, k=1)].max():.4f}")

# Build adjacency matrix: start with correlation
adj = corr.copy()

# Apply threshold to keep edges with correlation > 0.5 (you can adjust)
threshold = 0.5
adj_thresh = adj.where(adj > threshold, 0)
num_edges_corr = (adj_thresh.values > 0).sum() - len(adj_thresh)  # exclude diagonal
print(f"Number of correlation edges (threshold > {threshold}): {num_edges_corr}")

# Add sector edges: connect tickers in same sector with weight=1 (max)
num_sector_edges = 0
for t1 in adj_thresh.index:
    for t2 in adj_thresh.columns:
        if t1 != t2 and ticker_sector[t1] == ticker_sector[t2]:
            if adj_thresh.loc[t1, t2] == 0:
                adj_thresh.loc[t1, t2] = 1.0
                num_sector_edges += 1

print(f"Number of sector edges added: {num_sector_edges}")

# Final adjacency matrix stats
total_edges = (adj_thresh.values > 0).sum() - len(adj_thresh)
num_nodes = len(adj_thresh)
density = total_edges / (num_nodes * (num_nodes - 1))
print(f"Final graph stats:")
print(f"  Nodes: {num_nodes}")
print(f"  Edges: {total_edges}")
print(f"  Density: {density:.4f}")

# Save adjacency matrix as numpy file
np.save(adj_path, adj_thresh.values)
print(f"✅ Saved adjacency matrix to {adj_path}")

# Show adjacency matrix for visual check
adj_thresh
import networkx as nx
import matplotlib.pyplot as plt

# Create graph from adjacency matrix
G = nx.from_numpy_array(adj_thresh.values)
labels = {i: ticker for i, ticker in enumerate(adj_thresh.columns)}

# Define colors for sectors
sector_colors = {
    "Tech": "skyblue",
    "Consumer": "lightgreen"
}

# Map node colors based on sector
node_colors = [sector_colors[ticker_sector[t]] for t in adj_thresh.columns]

plt.figure(figsize=(10, 10))

# Spring layout for better spacing
pos = nx.spring_layout(G, seed=42)

# Draw nodes with colors
nx.draw_networkx_nodes(G, pos, node_color=node_colors, node_size=800, alpha=0.9)

# Draw edges with transparency and without self loops
edges = [(u, v) for u, v in G.edges() if u != v]
nx.draw_networkx_edges(G, pos, edgelist=edges, alpha=0.5)

# Draw labels
nx.draw_networkx_labels(G, pos, labels, font_size=12)

# Prepare stats text (you can update these variables if they are outside this code block)
stats_text = (
    f"Number of sector edges added: {num_sector_edges}\n"
    f"Nodes: {num_nodes}\n"
    f"Edges: {total_edges}\n"
    f"Density: {density:.4f}"
)

# Add stats text box to the plot (top left corner)
plt.gca().text(
    0.05, 0.95, stats_text,
    transform=plt.gca().transAxes,
    fontsize=12,
    verticalalignment='top',
    bbox=dict(boxstyle='round,pad=0.5', facecolor='white', alpha=0.8)
)

plt.title("Ticker Graph (Correlation + Sector Edges)")
plt.axis('off')
plt.show()



In [ ]:
import networkx as nx
import matplotlib.pyplot as plt

# Create graph from adjacency matrix
G = nx.from_numpy_array(adj_thresh.values)
labels = {i: ticker for i, ticker in enumerate(adj_thresh.columns)}

# Define colors for sectors
sector_colors = {
    "Tech": "skyblue",
    "Consumer": "lightgreen"
}

# Map node colors based on sector
node_colors = [sector_colors[ticker_sector[t]] for t in adj_thresh.columns]

plt.figure(figsize=(10, 10))

# Spring layout for better spacing
pos = nx.spring_layout(G, seed=42)

# Draw nodes with colors
nx.draw_networkx_nodes(G, pos, node_color=node_colors, node_size=800, alpha=0.9)

# Draw edges with transparency and without self loops
edges = [(u, v) for u, v in G.edges() if u != v]
nx.draw_networkx_edges(G, pos, edgelist=edges, alpha=0.5)

# Draw labels
nx.draw_networkx_labels(G, pos, labels, font_size=12)

plt.title("Ticker Graph (Correlation + Sector Edges)")
plt.axis('off')
plt.savefig("data/processed/ticker_graph.png", dpi=300, bbox_inches='tight')
plt.show()


In [ ]:
!pip install plotly networkx


import plotly.graph_objects as go
import networkx as nx

G = nx.from_pandas_adjacency(adj_thresh)
pos = nx.spring_layout(G, seed=42)

edge_x, edge_y = [], []
for edge in G.edges():
    x0, y0 = pos[edge[0]]
    x1, y1 = pos[edge[1]]
    edge_x += [x0, x1, None]
    edge_y += [y0, y1, None]

edge_trace = go.Scatter(x=edge_x, y=edge_y, mode='lines', line=dict(width=0.5, color='#888'), hoverinfo='none')

node_x, node_y, text = [], [], []
for node in G.nodes():
    x, y = pos[node]
    node_x.append(x)
    node_y.append(y)
    text.append(f"{node}<br>Sector: {ticker_sector[node]}")

node_trace = go.Scatter(
    x=node_x, y=node_y, mode='markers+text', text=text,
    marker=dict(size=12, color='skyblue', line_width=2),
    hoverinfo='text'
)

fig = go.Figure(data=[edge_trace, node_trace],
                layout=go.Layout(showlegend=False, hovermode='closest',
                                 margin=dict(b=0,l=0,r=0,t=40),
                                 title="Interactive Correlation Graph"))
fig.show()
